### GPU/TPU 需要固定形状的张量进行并行计算
- zero-padding解决负载不均衡导致的计算效率低下的关键技术
- 没有zero-padding，MoE无法再现代硬件上高效运行

In [1]:
import  torch
import torch.nn as nn

class MoELayerWithPadding(nn.Module):
    def __init__(self,num_experts,d_model,top_k=2,capacity_factor=1.0):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.d_model = d_model
        
        # 专家网络
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model,d_model * 4),
                nn.GELU(),
                nn.Linear(d_model * 4,d_model)
            ) for _ in range(num_experts)
        ])
        
        # 门控：选择哪个专家
        self.gate = nn.Linear(d_model,num_experts)
        # 容量限制： 每个专家最多处理的token数量
        self.capacity_factor = capacity_factor
    
    def forward(self,x):
        """
        x ： [batch_size,seq_len,d_model]
        """
        batch_size,seq_len,d_model = x.shape
        num_tokens = batch_size * seq_len
        x_flat = x.view(num_tokens,d_model) # [num_tokens,d_model]
        
        # 1.计算路由分数
        router_logits = self.gate(x_flat) # [num_tokens,num_experts]
        
        # 2.top-k 选择 ，topk_indices：每个token选的k个专家ID
        topk_logits,topk_indices = torch.topk(router_logits,self.top_k,dim=-1) # [num_tokens,top_k]
        # topk中专家的概率
        topk_probs = torch.softmax(topk_logits,dim=-1)
        
        # 3.计算每个庄家的容量（带padding）
        capacity = int(self.capacity_fator * num_tokens * self.top_k / self.num_experts)
        
        # 4 为每个专家收集token(带padding)
        expert_inputs = [] # 每个专家的输入[capacity,d_model]
        # [num_experts, capacity], 1=真实token, 0=padding
        expert_masks = [] # 标记哪些是padding(所有专家) 
        
        # 最终要变成规整的张量 [num_experts, capacity, d_model]，才能并行喂给所有专家
        for expert_id in  range(self.num_experts):
            # 找到选择当前专家的token
            token_mask = (topk_indices == expert_id).any(dim=-1)
            # 实际要送给这个专家处理的token
            selected_tokens = x_flat[token_mask] # [num_selected,d_model]
            num_selected = selected_tokens.size(0)
            
            # zero padding ：token不足->填充到固定容量
            if num_selected < capacity:
                padding = torch.zeros(
                    capacity - num_selected,d_model,device=x.device,dtype=x.dtype
                )
                expert_input = torch.cat([selected_tokens,padding],dim=0)
                # mask: 1表示真实token，0表示padding（后续计算要mask掉）
                mask = torch.cat([torch.ones(num_selected,device=x.device)],
                                torch.zeros(capacity - num_selected,device=x.device))
            else : # 超出容量->截断(或重路由
                """
                注意：被截断的token丢失了！这是Switch Transformer等原始MoE的做法，
                     更优的做法是overflow token重路由到第二个专家。
                """
                expert_input = selected_tokens[:capacity]
                mask = torch.ones(capacity,device=x.device)
            expert_inputs.append(expert_input)
            expert_masks.append(mask)
        
        # 5.并行计算所有专家(通过padding统一了长度)
        expert_outputs = []
        for expert_id in range(self.num_experts):
            # 每个专家的输出
            output = self.experts[expert_id](expert_inputs[expert_id])
            # 应用mask：将padding位置输出设置为0
            output = output * expert_mask[expert_id].unsqueeze(-1)
            expert_outputs.append(output)
            
        # 6.合并结果(取出padding)
        final_output = torch.zeros_like(x_flat)
        for token_idx in range(num_tokens):
            # topk
            for k in range(self.top_k):
                expert_id = topk_indices[token_idx,k].item()
                prob = topk_probs[token_idx,k]
                # 找到该token在专家输入中的位置(实际实现需要更复杂的索引)
                final_output[token_idx] += prob * expert_output_contribution
        return final_output.view(batch_size,seq_len,d_model)
                